# 01 — Exploratory Data Analysis

This notebook improves the original `Parkisons.ipynb` (load, overview, quality checks, target plot, gender plot).

Relationships described here are **observations in this dataset**, not medical causes.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
%matplotlib inline

## 2. Load Dataset

In [ ]:
candidates = [
    Path("../Dataset/parkinsons_disease_data.csv"),
    Path("../../ML-Project/parkinsons_disease_data.csv"),
]
DATA_PATH = next(p for p in candidates if p.exists())
print("Loading:", DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully")
print("Shape:", df.shape)

## 3. Dataset Overview

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe().T

## 4. Data Quality

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
for col in df.columns:
    print(col)
    uniques = df[col].unique()
    print(uniques[:25], "..." if len(uniques) > 25 else "")
    print("-" * 40)

In [ ]:
for name in ["Unnamed: 34", "Unnamed: 35"]:
    if name in df.columns:
        print(name, "non-null count:", df[name].notna().sum())

## 5. Data Cleaning

`Unnamed: 34` and `Unnamed: 35` are empty (trailing commas in the CSV). They are dropped.

`PatientID` is an identifier. It is kept for EDA and **must not** be used as an ML feature later.

`pd.get_dummies` is not needed: columns are already numeric or binary.

In [ ]:
unnamed_columns = [c for c in df.columns if str(c).startswith("Unnamed")]
print("Unnecessary empty columns:", unnamed_columns)

df = df.drop(columns=unnamed_columns)
print("Duplicate rows before drop:", df.duplicated().sum())
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)
print("Missing values after cleaning:", df.isnull().sum().sum())

## 6. Target Analysis

Target column: `Diagnosis`

In [ ]:
print(df["Diagnosis"].value_counts())
print("\nPercentage:")
print((df["Diagnosis"].value_counts(normalize=True) * 100).round(2))

In [ ]:
ax = sns.countplot(x="Diagnosis", data=df, palette="Set2")
plt.title("Diagnosis counts")
plt.xlabel("Diagnosis (0 = Negative, 1 = Positive)")
plt.ylabel("Number of records")
for c in ax.containers:
    ax.bar_label(c)
plt.show()

## 7. Univariate Analysis

In [ ]:
num_cols = [
    "Age", "BMI", "UPDRS", "MoCA", "FunctionalAssessment",
    "PhysicalActivity", "SleepQuality", "CholesterolTotal",
    "CholesterolLDL", "CholesterolHDL", "CholesterolTriglycerides",
]

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(col)
axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="lightblue")
    axes[i].set_title(col)
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 8. Categorical / Binary Analysis

In [ ]:
cat_cols = [
    "Gender", "Smoking", "Hypertension", "Diabetes", "Depression",
    "Stroke", "Tremor", "Rigidity", "Bradykinesia", "Posture",
    "SpeechProblems", "SleepDisorders", "Constipation",
]

fig, axes = plt.subplots(5, 3, figsize=(16, 18))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    sns.countplot(x=col, hue="Diagnosis", data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} vs Diagnosis")
for j in range(len(cat_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# AlcoholConsumption is continuous (units/week), not a 0/1 flag.
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="AlcoholConsumption", hue="Diagnosis", common_norm=False)
plt.title("AlcoholConsumption by Diagnosis")
plt.show()

## 9. Bivariate Analysis

In [ ]:
bivariate_num = ["Age", "BMI", "UPDRS", "MoCA"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(bivariate_num):
    sns.boxplot(x="Diagnosis", y=col, data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} vs Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
bivariate_bin = ["Tremor", "Rigidity", "Bradykinesia", "SleepDisorders"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(bivariate_bin):
    sns.countplot(x=col, hue="Diagnosis", data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} vs Diagnosis")
plt.tight_layout()
plt.show()

## 10. Correlation Analysis

In [ ]:
corr = df.drop(columns=["PatientID"]).corr()
plt.figure(figsize=(18, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.3)
plt.title("Correlation heatmap (PatientID excluded)")
plt.tight_layout()
plt.show()

In [ ]:
target_corr = corr["Diagnosis"].drop("Diagnosis").sort_values(key=abs, ascending=False)
print("Correlation with Diagnosis (strongest absolute values first):")
print(target_corr.head(15))

## 11. EDA Conclusions

```text
EDA CONCLUSION

- Dataset contains 2105 records and 36 raw columns (34 after dropping two empty Unnamed columns).
- Target variable is Diagnosis.
- Missing values: none in real feature columns. Unnamed: 34 and Unnamed: 35 were 100% empty.
- Duplicate rows: 0.
- Class counts in this file: 1304 Diagnosis=1 (about 62%) and 801 Diagnosis=0 (about 38%).
- PatientID is an identifier and should not be used as an ML feature.
- Important observations in THIS dataset (not causes):
  motor flags (Tremor, Rigidity, Bradykinesia, Posture) and scores (UPDRS, MoCA,
  FunctionalAssessment) show visible group differences on the plots.
- Features that appear useful for prediction here: UPDRS, MoCA, FunctionalAssessment,
  Tremor, Rigidity, Bradykinesia, Posture, and other symptom flags.
  Confirm with coefficients and 03_Model_Comparison.ipynb.
```
